# 02 - Agente Simples
> Construindo um agente funcional do zero

**Modulo:** `04_agentes` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import json

NOTAS={}

TOOLS=[
    {'name':'salvar_nota','description':'Salva uma nota.',
     'input_schema':{'type':'object','properties':{'titulo':{'type':'string'},'conteudo':{'type':'string'}},'required':['titulo','conteudo']}},
    {'name':'listar_notas','description':'Lista titulos das notas.',
     'input_schema':{'type':'object','properties':{}}},
    {'name':'buscar_nota','description':'Busca nota pelo titulo.',
     'input_schema':{'type':'object','properties':{'titulo':{'type':'string'}},'required':['titulo']}}
]

def salvar_nota(titulo,conteudo): NOTAS[titulo]=conteudo; return f'Nota "{titulo}" salva.'
def listar_notas(): return json.dumps(list(NOTAS.keys()),ensure_ascii=False)
def buscar_nota(titulo): return NOTAS.get(titulo,'Nao encontrada.')
FUNS={'salvar_nota':salvar_nota,'listar_notas':listar_notas,'buscar_nota':buscar_nota}

class Assistente:
    def __init__(self): self.hist=[]
    def chat(self,msg):
        self.hist.append({'role':'user','content':msg})
        while True:
            r=client.messages.create(model=HAIKU,max_tokens=512,tools=TOOLS,
                system='Voce gerencia notas.',messages=self.hist)
            if r.stop_reason=='end_turn':
                t=r.content[0].text; self.hist.append({'role':'assistant','content':t}); return t
            self.hist.append({'role':'assistant','content':r.content})
            res=[]
            for b in r.content:
                if b.type=='tool_use':
                    res.append({'type':'tool_result','tool_use_id':b.id,'content':str(FUNS[b.name](**b.input))})
            self.hist.append({'role':'user','content':res})

a=Assistente()
for m in ['Salva uma nota chamada Reuniao com: Discutir roadmap Q1.',
           'Quais notas eu tenho?','Me mostre a nota Reuniao.']:
    print(f'Voce: {m}'); print(f'Bot: {a.chat(m)}\n')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
